# Step 13 — Multi-Agent (Sequential)

🇬🇧 **English** (this notebook)

Add a second agent with a different, complementary role, and wire both into a `Crew` with `process=Process.sequential` — the same fixed pipeline this repo's full demo project (`src/research_crew/crew.py`) uses, just defined directly in this notebook instead of via `@CrewBase` and YAML config files.

This is the last **required** step in the exercise series — your final submission is due after this notebook. [Step 14](step_14_multi_agent_hierarchical.ipynb) revisits these same two agents under a different `Process` strategy (`hierarchical`, where a manager decides at runtime who does what); it's optional, worth exploring once your submission is in.</cell id="6fed75ac">


## Learning objective

By the end of this notebook, you will:

- Understand why a second, complementary agent can produce something neither agent produces alone
- Have chained two `Task`s together with `context=[...]`, so one agent's output feeds directly into another's input
- Have run your first real `Crew` with two agents and `process=Process.sequential`

## Prerequisites

- [Step 09 — Single Agent](step_09_single_agent.ipynb) completed — this notebook reuses its Researcher agent unchanged
- The same `.env` setup as the previous steps

## Background

A single agent can do multiple things, but it can't hold two genuinely different epistemic roles at the same time — it can't be simultaneously credulous (collecting everything) and skeptical (questioning what it found). Two agents let you encode that tension into the architecture. The seminal demonstration of agents collaborating through conversation was:

> Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)

## How this works

Two agents and two tasks, wired into a `Crew` with `process=Process.sequential`:

1. **`researcher`** is Step 09's exact same Researcher agent, unchanged.
2. **`analyst`** is a new agent with a different `role`/`goal`/`backstory` — someone who turns raw findings into a report for a specific audience, a job the Researcher was never asked to do.
3. **`research_task`** is assigned to the Researcher; **`analysis_task`** is assigned to the Analyst, with one crucial addition: `context=[research_task]`. This tells CrewAI to feed the first task's output into the second automatically — the same idea as Step 06's chain prompting, but wired declaratively instead of copy-pasted by hand.
4. **`crew.kickoff()`** runs both tasks in the order they appear in `tasks=[...]`, because `process=Process.sequential`.

This crew also sets `tracing=True` — [Step 02](step_02_intro_to_crewai.ipynb) introduced what that gives you over `verbose=True` — so the run prints its own shareable dashboard link; `crewai_event_bus.flush()` right after `kickoff()` just makes sure that link prints before the cell finishes. Neither call changes what the agents actually do, but with two agents handing work to each other, a trace view earns its keep here in a way it didn't yet in Step 02.

In [1]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process

load_dotenv()

topic = "EU AI Act"

# ── Agent 1: Researcher — same "researcher" template as agents.yaml ──────────
researcher = Agent(
    role=f"{topic} Senior Data Researcher",
    goal=f"Uncover cutting-edge developments in {topic}",
    backstory=(
        f"You're a seasoned researcher with a knack for uncovering the latest "
        f"developments in {topic}. Known for your ability to find the most relevant "
        f"information and present it in a clear and concise manner."
    ),
    verbose=True,
)

# ── Agent 2: Analyst — same "analyst" template as agents.yaml ─────────────────
analyst = Agent(
    role=f"{topic} Reporting Analyst",
    goal=f"Create detailed reports based on {topic} data analysis and research findings",
    backstory=(
        "You're a meticulous analyst with a keen eye for detail. You're known for "
        "your ability to turn complex data into clear and concise reports, making "
        "it easy for others to understand and act on the information you provide."
    ),
    verbose=True,
)

# ── Task 1: research — assigned to the Researcher ─────────────────────────────
research_task = Task(
    description=(
        f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
        "limited, minimal) and what obligations apply to providers of high-risk AI systems."
    ),
    expected_output=f"A structured summary of {topic}'s risk categories and obligations.",
    agent=researcher,
)

# ── Task 2: analysis — assigned to the Analyst, chained to Task 1 via `context` ─
analysis_task = Task(
    description=(
        "Using the research findings, write a short report for a product team that ships "
        "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
        "to them and by when."
    ),
    expected_output="A short, actionable report for a product team, grounded in the research findings.",
    agent=analyst,
    context=[research_task],
)

# ── Crew — same sequential process as src/research_crew/crew.py ──────────────
crew = Crew(
    agents=[researcher, analyst],
    tasks=[research_task, analysis_task],
    process=Process.sequential,
    verbose=True,
    # Prints a free, no-signup shareable trace URL (agent reasoning, task
    # timing, tool calls) to app.crewai.com after this cell finishes.
    tracing=True,
)

# ── Process — kick off the crew ───────────────────────────────────────────────
result = crew.kickoff()

# Trace upload happens on a background thread; a plain script naturally waits
# for it at process exit, but a notebook's kernel keeps running - flush() blocks
# until it's done, so the trace link below is guaranteed to print in this cell.
from crewai.events import crewai_event_bus
crewai_event_bus.flush()

print("=== Researcher output ===")
print(research_task.output.raw)
print("\n=== Analyst output ===")
print(analysis_task.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  519fc54b-e0f3-467d-8422-0d11858b0774                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│  ID: dc144927-304a-49ee-b60b-114a89a4519c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As a Senior Data Researcher specializing in the EU AI Act, I provide the following breakdown of the            │
│  risk-based framework. This architecture is the cornerstone of the legislation, designed to calibrate           │
│  regulatory oversight to the potential harm posed by an AI system.                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### I. The EU AI Act: Risk-Based Categories                                                                    │
│                                                                                                                 │
│  The Act categorizes AI systems into four distinct tiers based on their potential to impact the fundamental     │
│  rights, safety, and health of individuals.                                                                     │
│                                                                                                                 │
│  #### 1. Unacceptable Risk (Prohibited)                                                                         │
│  These systems are considered a clear threat to fundamental rights and are strictly **banned**.                 │
│  *   **Examples:** Social scoring systems by governments; AI used for real-time biometric identification in     │
│  public spaces by law enforcement (with narrow exceptions); AI that exploits vulnerabilities of specific        │
│  groups (age, disability); manipulative AI (subliminal techniques) that distorts behavior.                      │
│                                                                                                                 │
│  #### 2. High-Risk                                                                                              │
│  These systems are permitted but are subject to **stringent compliance requirements** before entering the EU    │
│  market. They are generally defined as those used in critical infrastructure, education, employment, essential  │
│  private/public services, and law enforcement.                                                                  │
│  *   **Examples:** AI in medical devices, critical transport infrastructure, recruitment processes (CV          │
│  filtering), or credit scoring.                                                                                 │
│                                                                                                                 │
│  #### 3. Limited Risk (Transparency)                                                                            │
│  These systems must adhere to **specific transparency obligations** to ensure users are aware they are          │
│  interacting with an AI.                                                                                        │
│  *   **Examples:** Chatbots, AI that generates or manipulates image, audio, or video content (e.g.,             │
│  Deepfakes). Users must be informed that they are interacting with a machine or that content is AI-generated.   │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what obligations     │
│  apply to providers of high-risk AI systems.                                                                    │
│  Agent:                                                                                                         │
│  EU AI Act Senior Data Researcher                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Reporting Analyst                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Internal Memo: Compliance Requirements for AI Hiring Tool under the EU AI Act**                              │
│                                                                                                                 │
│  **To:** Product Team                                                                                           │
│  **From:** EU AI Act Reporting Analyst                                                                          │
│  **Date:** May 22, 2024                                                                                         │
│  **Subject:** Mandatory Compliance Requirements for AI-Powered Recruitment Tools                                │
│                                                                                                                 │
│  ### 1. Classification                                                                                          │
│  Under the EU AI Act, your AI-powered hiring tool is classified as **High-Risk**. Systems used in recruitment,  │
│  specifically for CV filtering, assessing candidates, and decision-making in hiring processes, are explicitly   │
│  defined as high-risk due to their significant impact on fundamental rights and professional livelihoods.       │
│                                                                                                                 │
│  ### 2. Mandatory Compliance Obligations                                                                        │
│  As a provider of a high-risk AI system, you are legally required to fulfill the following obligations before   │
│  placing your product on the EU market:                                                                         │
│                                                                                                                 │
│  *   **Risk Management System:** Establish a continuous, documented process to identify and mitigate risks      │
│  throughout the AI lifecycle.                                                                                   │
│  *   **Data Governance:** Ensure training and validation datasets are high-quality, representative, and free    │
│  of bias regarding the intended geographical and behavioral context.                                            │
│  *   **Technical Documentation:** Compile detailed documentation (per Annex IV) demonstrating conformity with   │
│  the Act. This must be available for market surveillance authorities.                                           │
│  *   **Record-Keeping:** Implement automatic event logging to ensure full traceability of the AI’s              │
│  decision-making process.                                                                                       │
│  *   **Transparency & Information:** Provide clear instructions to clients (deployers) regarding the system’s   │
│  limitations, capabilities, and necessary human oversight.                                                      │
│  *   **Human Oversight:** Integrate features that allow human operators to effectively supervise, interpret,    │
│  override, or reverse AI outputs.                                                                               │
│  *   **Robustness & Cybersecurity:** Ensure the system is resilient against unauthorized manipulation and       │
│  maintains consistent accuracy throughout its lifecycle

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the research findings, write a short report for a product team that ships an AI-powered hiring tool in   │
│  the EU. Call out exactly which obligations apply to them and by when.                                          │
│  Agent:                                                                                                         │
│  EU AI Act Reporting Analyst                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 0ca1da29-459c-4d2f-a5e0-ba3d147d6bca                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/0ca1da29-459c-4d2f-a5e0-ba3d147d6bca?access_code=TRA │
│ CE-5402b9c06c                                                                                                   │
│ 🔑 Access Code: TRACE-5402b9c06c                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  519fc54b-e0f3-467d-8422-0d11858b0774                                                                           │
│  Final Output: **Internal Memo: Compliance Requirements for AI Hiring Tool under the EU AI Act**                │
│                                                                                                                 │
│  **To:** Product Team                                                                                           │
│  **From:** EU AI Act Reporting Analyst                                                                          │
│  **Date:** May 22, 2024                                                                                         │
│  **Subject:** Mandatory Compliance Requirements for AI-Powered Recruitment Tools                                │
│                                                                                                                 │
│  ### 1. Classification                                                                                          │
│  Under the EU AI Act, your AI-powered hiring tool is classified as **High-Risk**. Systems used in recruitment,  │
│  specifically for CV filtering, assessing candidates, and decision-making in hiring processes, are explicitly   │
│  defined as high-risk due to their significant impact on fundamental rights and professional livelihoods.       │
│                                                                                                                 │
│  ### 2. Mandatory Compliance Obligations                                                                        │
│  As a provider of a high-risk AI system, you are legally required to fulfill the following obligations before   │
│  placing your product on the EU market:                                                                         │
│                                                                                                                 │
│  *   **Risk Management System:** Establish a continuous, documented process to identify and mitigate risks      │
│  throughout the AI lifecycle.                                                                                   │
│  *   **Data Governance:** Ensure training and validation datasets are high-quality, representative, and free    │
│  of bias regarding the intended geographical and behavioral context.                                            │
│  *   **Technical Documentation:** Compile detailed documentation (per Annex IV) demonstrating conformity with   │
│  the Act. This must be available for market surveillance authorities.                                           │
│  *   **Record-Keeping:** Implement automatic event logging to ensure full traceability of the AI’s              │
│  decision-making process.                                                                                       │
│  *   **Transparency & Information:** Provide clear instructions to clients (deployers) regarding the system’s   │
│  limitations, capabilities, and necessary human oversight.                                                      │
│  *   **Human Oversight:** Integrate features that allow human operators to effectively supervise, interpret,    │
│  override, or reverse AI outputs.                     

=== Researcher output ===
As a Senior Data Researcher specializing in the EU AI Act, I provide the following breakdown of the risk-based framework. This architecture is the cornerstone of the legislation, designed to calibrate regulatory oversight to the potential harm posed by an AI system.

---

### I. The EU AI Act: Risk-Based Categories

The Act categorizes AI systems into four distinct tiers based on their potential to impact the fundamental rights, safety, and health of individuals.

#### 1. Unacceptable Risk (Prohibited)
These systems are considered a clear threat to fundamental rights and are strictly **banned**.
*   **Examples:** Social scoring systems by governments; AI used for real-time biometric identification in public spaces by law enforcement (with narrow exceptions); AI that exploits vulnerabilities of specific groups (age, disability); manipulative AI (subliminal techniques) that distorts behavior.

#### 2. High-Risk
These systems are permitted but are subject to **st

## Your task

1. Run the cell. Compare the Analyst's report to Step 09's Researcher answer alone — is it more targeted and actionable, or does it mostly repackage the same content?

2. **Experiment**: remove `context=[research_task]` from `analysis_task` and rerun. Without that link, does the Analyst still somehow reference the Researcher's specific findings, or does it write a generic report from scratch?

3. Now swap in your own team's topic — keep the Researcher agent from Step 09, and give it a second, genuinely complementary role and task suited to your topic.

4. This is where your project shifts from guided exercises to your own design: use everything from Steps 03–13 to build and document your own agent. Fill in `REPORT.md` — see [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full report structure and what's expected. Make sure the **Sprint Progression** table has all four rows filled in — it's the fastest way for your reader to see the arc before they read the rest of the report.

## Shortcomings

The pipeline is fixed: the Researcher always runs first, the Analyst always runs second, and both *order* and *who does what* are wired into the code via `agent=...` — nothing about it adapts to what either agent actually finds. [Step 14](step_14_multi_agent_hierarchical.ipynb) explores one way to loosen that: a manager agent decides *who* handles each task at runtime, though even there the task *order* stays fixed.

Neither agent above uses any of the tools/MCP/RAG grounding from Steps 10–12 — combining multi-agent orchestration with a grounded, tool-using Researcher is a natural next experiment, just not one this notebook walks you through.

This is the last required step in the exercise series. From here, the main [README's use-case table](../../README.md#use-cases-to-pick-from) and `REPORT.md` ask you to step back and judge, for your own topic, which combination of everything covered — single vs. multi-agent, tools, MCP, RAG — is actually worth the added complexity.

## Resources for further reading

- Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)
- [CrewAI `Process` concept docs](https://docs.crewai.com/en/concepts/processes) — `sequential` vs. `hierarchical`, covered briefly in Step 02 and in depth in [Step 14](step_14_multi_agent_hierarchical.ipynb)

## Stretch goal

Add a third agent whose absence you'd actually notice — a critic, a translator for a non-expert audience, or a validator. Give it its own `Task`, chained in with `context=[...]` like `analysis_task` is above. Does the output meaningfully change? If it doesn't, ask yourself why.

For a different kind of stretch, [Step 14](step_14_multi_agent_hierarchical.ipynb) is the more natural fit for a third agent whose *involvement* — not just its output — should vary run to run: a manager can bring it in only when needed, something a fixed `agent=...` pipeline can't express.

---

**This is your final submission.** See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full grading rubric and exactly what to submit.